# Build the best associative memory you can, with scaling laws

You have **10¹³ flops** and **3 screening rounds** to decide how to spend them.
Then you get **one shot** at a final "hero" run. No do-overs.

### The model
$\hat p(\cdot\mid x) = \mathrm{softmax}(W e_x)$, with $W \in \mathbb{R}^{512\times n}$ and no bias.
The output dimension is fixed at $d=512$; $e_x$ is a random unit vector attached to token $x$.
So the model has $N = 512n$ parameters and **its only job is to memorise $p(y\mid x)$ for as many $x$ as it can.**

### The data
* inputs: $p(x)\propto x^{-1.2}$, cut off at $p=10^{-14}$ — that is a vocabulary of $1.1\times10^{11}$ tokens,
  so you will never see most of them, and the ones you do see follow a brutally heavy tail;
* outputs: for each $x$, a fixed random distribution over the 512 classes whose effective
  support $e^{H(x)}$ averages 16.

### What you control
| knob | notes |
|---|---|
| `n` — the width | the *only* size knob. $N = 512n$ |
| `steps` (or `c`, the flops per run) | tokens $D = 64\times$ steps |
| `lr` — peak learning rate | cosine decay to `lr/10`, no warmup |

Fixed by the rules: Adam, batch 64, cross-entropy loss, cosine schedule.

### Flop accounting
Training costs $6ND$; every evaluation pass costs $2NM$ and **also counts**. Both are billed
automatically — you cannot accidentally overspend, the library refuses instead.

### Your deliverable
A recipe, the hero-run parameters, the loss you **predicted before running it**, and the loss you got.

## 0. Setup

Run `uv run assocmem-prepare` once in a terminal first (it builds ~250 MB of cached data, ~1 min).

Everything you do lives in `runs/<your-name>/` and survives a kernel restart — including the
flops you have already spent. `lab.reset(confirm=True)` starts over from scratch.

In [ ]:
%config InlineBackend.figure_formats = ['retina']
from assocmem import Lab, Sweep, BudgetError
from assocmem.plots import plot_summary

lab = Lab("me", budget=1e13, rounds=3)
print(f"irreducible loss (mean conditional entropy) = {lab.l_inf:.4f} nats")
print(f"predicting uniform would give log(512)      = 6.2383 nats")

## 1. How the API works

A `Sweep` is a cartesian product. Give the time axis as `c` (flops per run — steps are
derived from `n`) or as `steps` directly. `c` is what you want for scaling laws, because it
puts every run in the sweep at **equal compute**:

```python
Sweep(c=[4e9, 1.2e10], n=[64, 128, 256], lr=[0.03, 0.06])   # 2*3*2 = 12 runs
Sweep(...) + Sweep(...)                                     # concatenate, for irregular designs
```

Three rules to internalise:

1. **`sweep.estimate(lab)` is free.** Always price a sweep before running it.
2. **`lab.run_round(...)` spends one of your three rounds**, however many configs are in it.
   So put everything you want to learn *simultaneously* into one sweep.
3. **Re-running an identical config is free** and does not burn a round — results are cached.
   And `smoke=True` runs your sweep at 1% of the steps to check it works, for ~1% of the cost
   and no round.

In [ ]:
s = Sweep(c=[4e9, 1.2e10], n=[64, 128, 256, 512], lr=[0.0125, 0.025, 0.05, 0.1, 0.2])
s.estimate(lab)   # free: prints cost, share of budget, wall-clock estimate, and what it would cut

In [ ]:
# the library refuses instead of overspending -- and tells you what to cut
try:
    lab.run_round("much too big", Sweep(c=[3e12], n=[500, 1000, 2000], lr=[0.02, 0.04, 0.08]))
except BudgetError as e:
    print(e)

## 2. Round 1 — where is the learning rate, and what does an IsoFLOP curve look like?

At fixed compute `c`, sweeping `n` trades **capacity against data**: a wider model memorises
more per token seen, but takes fewer steps for the same flops. The loss-vs-`n` curve therefore
has a minimum — that is the *IsoFLOP optimum* $n^*(C)$, and it is the thing you are going to
extrapolate.

Round 1 should be **cheap and wide**: you do not know where `lr*` is yet, so bracket it
generously (a factor of ~16 from end to end), and use two cheap compute rungs so you get your
first two points on the $n^*(C)$ curve for free.

The sweep below is a reasonable starting point. Run it as-is the first time.

In [ ]:
s1 = Sweep(c=[4e9, 1.2e10],
           n=[64, 128, 256, 512],
           lr=[0.0125, 0.025, 0.05, 0.1, 0.2])

lab.run_round("R1 lr landscape", s1, smoke=True, plot=False)   # free of rounds: does it run?

In [ ]:
r1 = lab.run_round("R1 lr landscape", s1)

**Read the plots.** Before moving on, answer these from the figure:

1. Is `lr*` inside your grid, or pinned at an edge? If it is at an edge you have measured a
   *bound*, not an optimum, and the loss at that rung is an over-estimate.
2. Does `lr*` depend on `n`, or only on `c`? (Compare cells at the same `c` but different `n`.)
3. How **flat** is the IsoFLOP minimum? If being 2× wrong in `n` costs 0.03 nats but being 2×
   wrong in `lr` costs 0.06, you know which one deserves round 3.
4. `lab.fit()` needs ≥3 widths at ≥2 compute rungs. You already have that — try it.

In [ ]:
laws1 = lab.fit()
print("\nrecipe this law suggests for C = 1e12:", laws1.recipe(1e12))

## 3. Round 2 — extend the ladder

You are going to extrapolate ~20× beyond your most expensive rung, so **span matters more
than density**: rungs 3× apart teach you far more about the exponent than rungs 1.2× apart.

But cost scales with the rung: a profile of $k$ widths at compute $C$ costs $kC$. So the
arithmetic is unforgiving — spend few configs at expensive rungs, many at cheap ones.

⚠️ The defaults below are deliberately mediocre. **Edit them.** Think about:
* how high a rung can you afford, given you still need round 3 *and* a hero run?
* how many lrs do you need here, now that you know something about `lr*`?
* `laws1.lr(c)` will predict a learning rate for you — do you trust it?

In [ ]:
# ---------------- EDIT ME ----------------
s2 = Sweep(c=[4e10, 1.2e11],
           n=[128, 256, 512, 1024],
           lr=[0.025, 0.05])
# ----------------------------------------
s2.estimate(lab)

In [ ]:
r2 = lab.run_round("R2 extend the ladder", s2)
laws2 = lab.fit()

## 4. Round 3 — spend it on your biggest uncertainty

You have one round left. Two things are still uncertain, and they are not equally important:

* **the exponent of $n^*(C)$** — but the IsoFLOP minimum is flat, so guessing `n` wrong by 30%
  is nearly free;
* **`lr*` at the hero scale** — where the penalty for being wrong is much steeper, and which
  you have only measured at small scale.

A useful trick: at every rung so far, `lr*` was the same for *every* `n`. If that holds, you
can sweep `lr` at one width and reuse it at the others, which buys you a full IsoFLOP profile
**and** an lr parabola at your top rung for the price of ~5 runs.

⚠️ Again, the default below is mediocre — it has no lr sweep at all, so `lab.fit()` will have
to extrapolate `lr*` blind. **Fix that.**

In [ ]:
# ---------------- EDIT ME ----------------
c3 = 3e11
s3 = Sweep(c=[c3], n=[280, 560, 1120], lr=[laws2.lr(c3)])
# hint: + Sweep(c=[c3], n=[560], lr=[laws2.lr(c3) / 1.75, laws2.lr(c3) * 1.75])
# ----------------------------------------
s3.estimate(lab)

In [ ]:
r3 = lab.run_round("R3 top rung", s3)

## 5. Fit the laws

`lab.fit()` does three things: takes the per-rung IsoFLOP optimum $(n^*, L^*)$ from a parabola
in $\log n$, fits power laws $n^*(C)$, $\mathrm{lr}^*(C)$ and $L^*(C) = L_\infty + AC^{-\alpha}$,
and corrects any rung whose best `lr` sat away from the fitted `lr*` (it tells you when it does).

Check the $r^2$ values and the residuals. If a law is not clean, say so in your write-up — an
honest error bar is worth more than a confident wrong number.

In [ ]:
laws = lab.fit()
print("\n" + "-" * 70)
print("recipe at the compute you have left:", laws.recipe(lab.remaining))

## 6. The hero run — commit your prediction first

`lab.hero(laws)` sizes the run to every flop you have left, **prints the loss it predicts**,
and only then trains. That ordering is the point: a scaling law that you consult after seeing
the answer has taught you nothing.

You get one call. It will refuse a second.

In [ ]:
hero = lab.hero(laws)

In [ ]:
plot_summary(lab, laws)

## 7. Write it up

One page:

1. **Recipe** — the three laws, with $r^2$, and how you split your budget across rounds.
2. **Hero-run parameters** — `n`, steps, tokens, `lr`, and the flops it used.
3. **Expected loss** — what you predicted, and your uncertainty on it.
4. **Actual loss** — what you got, and *why the difference has the sign it does*.

Things worth commenting on:

* Your compute split $N\propto C^{b}$, $D\propto C^{1-b}$. How does $b$ compare to 0.5? How many
  tokens per parameter did your hero run use — and how does that compare to the ~20 of LLM
  scaling laws? Why is this problem in the opposite regime?
* The excess loss $L-L_\infty$ decays *very* slowly in $C$. Can you explain that from the Zipf
  tail? (Hint: if a model can memorise the top $K$ tokens, the mass it gets wrong is
  $\sum_{x>K}p(x)\propto K^{-(\gamma-1)}$ with $\gamma=1.2$ — a tiny exponent. What does that
  imply for $n^*\propto C^{1/(1+\gamma)}$?)

### The one number that matters most

Three reference attempts at this exercise, same problem instance and same eval set:

| attempt | spent on tuning | law quality | hero compute | hero loss |
|---|---|---|---|---|
| careful: 5 rungs, lr parabola at the top only | 47 % | $r^2=0.9992$ | $5.0\times10^{12}$ | 3.2993 |
| the mediocre defaults in this notebook | 26 % | $r^2=0.9983$ | $7.0\times10^{12}$ | 3.2765 |
| cheap wide ladder, lr parabola at every rung | **8 %** | $r^2=0.9997$ | $8.8\times10^{12}$ | **3.2609** |

**The cheapest one won.** All three hero runs — different $n$, different $\eta$, from
independently fitted laws — land on a single power law $L-L_\infty = 9.48\,C^{-0.083}$ with
residuals of 0.0001 nats. The hero loss is set by *the compute it receives*, almost regardless of
the recipe. Since $\alpha\approx0.083$, $10^{12}$ spent screening costs ~0.008 nats, while being
20 % wrong in $n$ costs 0.0014 and 30 % wrong in $\eta$ costs 0.006.

So the skill is not fitting the prettiest law. It is **getting $\eta$ right, ignoring $n$, and
screening as cheaply as the laws allow.** The ceiling with free tuning is 3.2552; the whole
spread between a careless and a perfect attempt is ~0.05 nats. Say that in your write-up, with an
error bar — and get the *sign* of your prediction error right, which matters more than the fourth
decimal place.

**Bonus, if you want to go further.** Your fitted $\alpha$ will over-predict the improvement at
the hero scale. Can you see why from the theory, and derive $\alpha$ from your measured width
exponent $b$ instead of fitting it? (Start from: capacity $\propto n^{c}$, excess $\propto
K^{-(\gamma-1)}$, and $b=1/(1+c\gamma)$.)